# Baseline Model — Naive Association (OLS)

The simplest baseline: a linear regression of energy intensity on the treatment indicator
`efficient_windows`. Two variants:

- **A) Univariate:** treatment only → the *naive correlation*.
- **B) Multivariate:** adjusts for the observed confounders (size, age, climate, housing type,
  household size, income).

The point: even the simple multivariate OLS already removes most of the confounding bias. But it
imposes (i) linearity and (ii) a single constant effect for every household — which the Causal
Forest (step 3) relaxes.

In [1]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
df = pd.read_csv('../data/recs2020_processed.csv')
print('n =', len(df))

n = 17196


## Model A — univariate (the naive correlation)

In [2]:
mA = smf.ols('energy_intensity_kwh_m2 ~ efficient_windows', data=df).fit()
print(mA.summary().tables[1])
print(f'\nNaive treatment coefficient: {mA.params["efficient_windows"]:+.2f} kWh/m2.a   '
      f'R2={mA.rsquared:.3f}')
print('   => Efficient-window homes use ~12 kWh/m2 less - but this is confounded.')

                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept           158.2099      1.088    145.373      0.000     156.077     160.343
efficient_windows   -12.1355      1.322     -9.179      0.000     -14.727      -9.544

Naive treatment coefficient: -12.14 kWh/m2.a   R2=0.005
   => Efficient-window homes use ~12 kWh/m2 less - but this is confounded.


## Model B — multivariate (adjust for observed confounders)

In [3]:
mB = smf.ols('energy_intensity_kwh_m2 ~ efficient_windows + floor_area_m2 + building_age '
              '+ C(climate_zone) + C(housing_type) + household_size + income_bin', data=df).fit()
print(mB.summary().tables[1].as_text()[:1600])
print(f'\nAdjusted treatment coefficient: {mB.params["efficient_windows"]:+.2f} kWh/m2.a   '
      f'p={mB.pvalues["efficient_windows"]:.4f}   R2={mB.rsquared:.3f}')
print(f'   => After adjustment the effect shrinks by ~{100*(1-abs(mB.params["efficient_windows"]/mA.params["efficient_windows"])):.0f}%'
      f' - most of the apparent saving was confounding.')

                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                        168.4820      3.215     52.404      0.000     162.180     174.784
C(climate_zone)[T.Hot-Dry]       -43.0177      2.015    -21.353      0.000     -46.966     -39.069
C(climate_zone)[T.Hot-Humid]     -48.9345      1.731    -28.271      0.000     -52.327     -45.542
C(climate_zone)[T.Marine]        -43.5001      2.534    -17.165      0.000     -48.467     -38.533
C(climate_zone)[T.Mixed-Dry]     -10.0095      6.034     -1.659      0.097     -21.836       1.817
C(climate_zone)[T.Mixed-Humid]   -18.8049      1.285    -14.632      0.000     -21.324     -16.286
C(climate_zone)[T.Subarctic]      74.8415     11.099      6.743      0.000      53.087      96.596
C(climate_zone)[T.Very-Cold]      30.5778      3.175      9.632      0.000      24.355      36.800
C(housing_

## 5-fold cross-validated predictive R²

In [4]:
X = pd.get_dummies(df[['efficient_windows','floor_area_m2','building_age','climate_zone',
                          'housing_type','household_size','income_bin']], drop_first=True).astype(float)
y = df['energy_intensity_kwh_m2']
cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print(f'CV R2: {cv.mean():.3f} +/- {cv.std():.3f}')

CV R2: 0.276 +/- 0.015


## Evaluation & practical relevance

| Model | Treatment coefficient | Reading |
|---|---|---|
| A — univariate | **≈ −12 kWh/m²·a** | overstates the true effect (confounded by age/climate/size) |
| B — multivariate OLS | **≈ −4 kWh/m²·a, p<0.001** | most of the apparent saving was confounding |

**Why the baseline is still not enough.** Multivariate OLS gives one constant effect for every
household. An energy manager who wants to target retrofits needs the *per-household* effect — which
buildings benefit most? That requires the Causal Forest (step 3).